# 📊 Atividade 1.2 — Desafio Resolvido

## 🖥️ Matplotlib, Seaborn e Desempenho de GPUs

### 🎯 Apresentação

Aluno: Carlos Ernesto Martins Vieira Neto

Matrícula: 202302530

Este notebook apresenta o estudo sobre dados de benchmarks de GPUs para analisar desempenho, preço, consumo energético e possíveis relações com cargas computacionais de inteligência artificial.

---


## 🎯 Exercícios Práticos

### 💪 **Desafio Final:**
Escolha um dataset do Kaggle e realize uma análise completa:
1. 🧹 **Limpeza dos dados**
2. 📊 **Análise exploratória**
3. 📈 **Visualizações informativas**
4. 🔍 **Identificação de insights**
5. 📋 **Conclusões e recomendações**


## ✅ Solução do Desafio Final

### 🖥️ Dataset escolhido: GPU Benchmarks Compilation (Kaggle)

O dataset reúne resultados de desempenho de GPUs. A solução realiza a limpeza dos registros, cria métricas de custo e desempenho e produz visualizações sobre poder computacional e eficiência energética.


In [ ]:
# Importando as bibliotecas necessárias
import io
import zipfile
import urllib.request
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_style('whitegrid')

# Baixando o dataset diretamente do Kaggle
url_gpu = 'https://www.kaggle.com/api/v1/datasets/download/alanjo/gpu-benchmarks'
requisicao_gpu = urllib.request.Request(url_gpu, headers={'User-Agent': 'Mozilla/5.0'})
with urllib.request.urlopen(requisicao_gpu) as resposta:
    conteudo_zip = resposta.read()

with zipfile.ZipFile(io.BytesIO(conteudo_zip)) as arquivo_zip:
    with arquivo_zip.open('GPU_benchmarks_v7.csv') as arquivo_csv:
        df_desafio = pd.read_csv(arquivo_csv)

print("🖥️ Dataset GPU Benchmarks carregado com sucesso!")
print(f"📏 Dimensões originais: {df_desafio.shape}")
print(f"🔁 Linhas duplicadas: {df_desafio.duplicated().sum()}")
print()
print("🧹 Valores ausentes antes da limpeza:")
print(df_desafio.isnull().sum()[df_desafio.isnull().sum() > 0])

# Renomeando as colunas para facilitar a análise
df_desafio = df_desafio.rename(columns={
    'gpuName': 'modelo_gpu',
    'G3Dmark': 'desempenho_3d',
    'G2Dmark': 'desempenho_2d',
    'price': 'preco_usd',
    'gpuValue': 'valor_gpu',
    'TDP': 'tdp_watts',
    'powerPerformance': 'eficiencia_energetica',
    'testDate': 'ano_teste',
    'category': 'categoria'
})

# Removendo duplicatas e padronizando textos
df_desafio = df_desafio.drop_duplicates().copy()
df_desafio['modelo_gpu'] = df_desafio['modelo_gpu'].str.strip()
df_desafio['categoria'] = df_desafio['categoria'].replace('Unknown', 'Não informado')
df_desafio['categoria'] = df_desafio['categoria'].str.strip().str.title()

# Garantindo que as medidas estejam no formato numérico
colunas_numericas = [
    'desempenho_3d', 'desempenho_2d', 'preco_usd',
    'valor_gpu', 'tdp_watts', 'eficiencia_energetica', 'ano_teste'
]
for coluna in colunas_numericas:
    df_desafio[coluna] = pd.to_numeric(df_desafio[coluna], errors='coerce')

# Mantendo apenas registros completos nas análises de preço e energia
df_analise = df_desafio.dropna(subset=[
    'preco_usd', 'tdp_watts', 'valor_gpu', 'eficiencia_energetica'
]).copy()

# Criando novas colunas derivadas
df_analise['custo_por_ponto'] = (
    df_analise['preco_usd'] / df_analise['desempenho_3d']
)
df_analise['faixa_desempenho'] = pd.qcut(
    df_analise['desempenho_3d'], q=4,
    labels=['Entrada', 'Intermediária', 'Avançada', 'Extrema']
)

print()
print("✅ Limpeza concluída!")
print(f"📊 GPUs disponíveis para análise geral: {len(df_desafio)}")
print(f"⚡ GPUs com preço e consumo informados: {len(df_analise)}")


### 📊 Análise Exploratória

As estatísticas descritivas e os agrupamentos permitem comparar desempenho, preço, consumo e categorias antes da criação dos gráficos.


In [ ]:
# Visualizando as primeiras linhas e as estatísticas descritivas
display(df_desafio.head())

print("📊 Estatísticas de desempenho, preço e consumo:")
display(df_analise[[
    'desempenho_3d', 'desempenho_2d', 'preco_usd',
    'tdp_watts', 'eficiencia_energetica', 'custo_por_ponto'
]].describe().round(2))

# Identificando as GPUs com maior desempenho 3D
top_desempenho = df_desafio.nlargest(10, 'desempenho_3d')[
    ['modelo_gpu', 'desempenho_3d', 'ano_teste', 'categoria']
]

# Comparando o desempenho mediano entre categorias conhecidas
desempenho_categoria = (
    df_desafio[df_desafio['categoria'] != 'Não Informado']
    .groupby('categoria')['desempenho_3d']
    .median()
    .sort_values(ascending=False)
)

print("🏆 Dez GPUs com maior desempenho 3D:")
display(top_desempenho)

print("🗂️ Desempenho 3D mediano por categoria:")
display(desempenho_categoria.round(2).to_frame('Mediana 3D'))


### 📈 Visualizações Informativas

Foram construídos quatro gráficos para comparar consumo, desempenho, evolução anual e correlações entre as principais variáveis do hardware.


In [ ]:
# Criando quatro visualizações sobre desempenho e eficiência
fig, eixos = plt.subplots(2, 2, figsize=(18, 13))

dados_categorias = df_analise[df_analise['categoria'] != 'Não Informado']
sns.scatterplot(
    data=dados_categorias, x='tdp_watts', y='desempenho_3d',
    hue='categoria', alpha=0.75, s=70, ax=eixos[0, 0]
)
eixos[0, 0].set_title('⚡ Consumo Energético vs. Desempenho 3D')
eixos[0, 0].set_xlabel('TDP (watts)')
eixos[0, 0].set_ylabel('Pontuação G3DMark')
eixos[0, 0].legend(title='Categoria')

top_gpus = df_desafio.nlargest(10, 'desempenho_3d').sort_values('desempenho_3d')
eixos[0, 1].barh(
    top_gpus['modelo_gpu'], top_gpus['desempenho_3d'],
    color='#2E86AB'
)
eixos[0, 1].set_title('🏆 GPUs com Maior Desempenho 3D')
eixos[0, 1].set_xlabel('Pontuação G3DMark')
eixos[0, 1].set_ylabel('Modelo')

evolucao_desempenho = df_desafio.groupby('ano_teste')['desempenho_3d'].max()
eixos[1, 0].plot(
    evolucao_desempenho.index, evolucao_desempenho.values,
    marker='o', color='#52B788', linewidth=2
)
eixos[1, 0].set_title('📈 Maior Desempenho Registrado por Ano')
eixos[1, 0].set_xlabel('Ano do Teste')
eixos[1, 0].set_ylabel('Maior Pontuação G3DMark')
eixos[1, 0].grid(True, alpha=0.3)

correlacoes_hardware = df_analise[[
    'desempenho_3d', 'preco_usd', 'tdp_watts',
    'eficiencia_energetica', 'valor_gpu'
]].corr()
sns.heatmap(
    correlacoes_hardware, annot=True, fmt='.2f',
    cmap='coolwarm', center=0, ax=eixos[1, 1]
)
eixos[1, 1].set_title('🔥 Correlações entre Desempenho, Preço e Energia')

plt.tight_layout()
plt.show()


### 🔍 Insights, Conclusões e Recomendações

Os resultados finais destacam desempenho, eficiência e custo-benefício. Também são apresentadas as limitações do benchmark e sua relação com a escolha de GPUs para projetos de inteligência artificial.


In [ ]:
# Selecionando GPUs modernas e de alto desempenho para comparações justas
limite_desempenho = df_analise['desempenho_3d'].quantile(0.75)
gpus_modernas = df_analise[
    (df_analise['ano_teste'] >= 2018) &
    (df_analise['desempenho_3d'] >= limite_desempenho)
]

gpu_mais_rapida = df_desafio.nlargest(1, 'desempenho_3d').iloc[0]
gpu_mais_eficiente = gpus_modernas.nlargest(1, 'eficiencia_energetica').iloc[0]
gpu_melhor_valor = gpus_modernas.nlargest(1, 'valor_gpu').iloc[0]

correlacao_preco = df_analise['preco_usd'].corr(df_analise['desempenho_3d'])
correlacao_tdp = df_analise['tdp_watts'].corr(df_analise['desempenho_3d'])

print("🔍 PRINCIPAIS INSIGHTS")
print(f"✅ Maior desempenho 3D: {gpu_mais_rapida['modelo_gpu']} ({gpu_mais_rapida['desempenho_3d']:.0f} pontos)")
print(f"✅ Melhor eficiência entre GPUs modernas e rápidas: {gpu_mais_eficiente['modelo_gpu']} ({gpu_mais_eficiente['eficiencia_energetica']:.2f} pontos/W)")
print(f"✅ Melhor relação desempenho/preço nesse grupo: {gpu_melhor_valor['modelo_gpu']} ({gpu_melhor_valor['valor_gpu']:.2f})")
print(f"✅ Correlação entre preço e desempenho: {correlacao_preco:.2f}")
print(f"✅ Correlação entre TDP e desempenho: {correlacao_tdp:.2f}")

print()
print("📋 CONCLUSÕES")
print("✅ Maior consumo energético costuma acompanhar maior desempenho, mas não garante eficiência.")
print("✅ Preço, desempenho bruto e desempenho por watt representam critérios diferentes.")
print("✅ A evolução anual mostra o crescimento do poder gráfico disponível.")

print()
print("🤖 RELAÇÃO COM INTELIGÊNCIA ARTIFICIAL")
print("✅ GPUs mais rápidas e eficientes podem reduzir o tempo e o consumo energético de cargas computacionais.")
print("⚠️ O G3DMark mede desempenho gráfico geral, não treinamento ou inferência de modelos de IA.")

print()
print("🎯 RECOMENDAÇÕES")
print("✅ Avaliar desempenho, consumo energético e preço antes da escolha da GPU.")
print("✅ Para projetos de IA, considerar também VRAM, CUDA/ROCm e suporte a FP16/BF16.")
print("✅ Utilizar benchmarks específicos da aplicação antes de adquirir o hardware.")
print("⚠️ O dataset reúne resultados até 2022 e não representa os lançamentos mais recentes.")
